# 五係數校準模擬：BOUND NN＋GP vs MLE＋GP

這份 notebook **只讀取結果，不會啟動訓練、校準或長時間 GP**。兩條流程使用同一份模擬 50 年年最大值、相同 GRID、seed、候選變數、kernel、folds 與 buffer；各自在內層選模，外層輸出 OOF。

| 比較層次 | NN 路線 | MLE 路線 | 誤差參考 |
|---|---|---|---|
| 估計器 | 固定 21D BOUND NN | 每格重新配適五係數 MLE | 已知模擬係數 |
| 完整空間流程 | NN → nested buffered GP | MLE → nested buffered GP | 相同已知模擬係數 |

五係數為 $\mu_0,\beta_\mu,\eta_0=\log\sigma_0,\beta_\sigma,\xi_0$。不逐格做四種時間結構選模，不修改正式 NN。正的改善率表示 NN RMSE 較低，負值表示 MLE 較低；不預設哪個方法一定較好。


In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# 搬機後可設定結果位置；不是原始每日檔的資料夾。
# RESULT_ROOT = Path(r"F:\論文跑\outputs\simulations\eas_calibrated_v1\mle_comparison_v2_crps")
RESULT_ROOT = None
cwd = Path.cwd()
if RESULT_ROOT is None:
    candidates = [p / prefix / version
                  for version in ["mle_comparison_v2_crps", "mle_comparison_v1"]
                  for prefix in ["outputs/simulations/eas_calibrated_v1", "results/simulation/eas_calibrated_v1"]
                  for p in [cwd, *cwd.parents]]
    found = [p for p in candidates if (p / "comparison_protocol.json").is_file()]
    RESULT_ROOT = found[0] if found else candidates[0]
RESULT_ROOT = Path(RESULT_ROOT)
def read_json(name):
    path = RESULT_ROOT / name
    return json.loads(path.read_text(encoding="utf-8-sig")) if path.exists() else {}
def read_csv(name):
    path = RESULT_ROOT / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()
protocol = read_json("comparison_protocol.json")
status = read_json("run_status.json")
accuracy = read_csv("parameter_comparison.csv")
timing = read_csv("timing_comparison.csv")
quality = read_csv("quality_comparison.csv")
display(Markdown(f"結果位置：{RESULT_ROOT}"))
if not protocol:
    display(Markdown("**尚未找到正式比較結果。** 設定 RESULT_ROOT，或完成下方的一次比較指令後重跑 notebook。這不代表 RMSE 或耗時是 0。"))
else:
    display(pd.DataFrame([status]))
    if status.get("status") not in ("completed", "completed_with_warnings"):
        display(Markdown("**尚未全部完成：以下若有表格，只包含已完成的成對 replicate。失敗或未完成的部分不可當成成功結果。**"))
    report = read_json("run_report.json")
    if status.get("status") in ("completed", "completed_with_warnings"):
        for name, digest in report.get("output_sha256", {}).items():
            actual = hashlib.sha256((RESULT_ROOT / name).read_bytes()).hexdigest()
            if actual != digest:
                raise ValueError(f"輸出被更動：{name}")


## 1. 確認比較條件

- 使用固定 NN＋GP 校準得到的五張參數曲面作為真值；每次重新生成 50 年資料，不是同一份真實資料重跑。
- MLE 初始化只使用該格觀測，不使用 NN、真值或情境標籤。
- 預設 **wide-domain bounded MLE**：四個 data-only 起始點、SLSQP、support 限制、必要時重試。不是無界 MLE，也不保證全域最優。
- 預設 MLE 用 1 worker；GP 候選平行度由原設定控制。調整設定須另開實驗，不混用計時。
- 此空間模擬**不是**先前 10,000 組四情境 test 表，不能混填結果。


In [ ]:
if protocol:
    config = protocol["config"]
    conditions = {
        "GRID count": accuracy.n_grids.iloc[0] if not accuracy.empty else "尚無完成 pair",
        "Years": f'{protocol["years"][0]}–{protocol["years"][-1]}',
        "Simulation seed": protocol["seed"],
        "Outer / inner folds": f'{config["outer_folds"]} / {config["inner_folds"]}',
        "Buffers km": config["buffers_km"],
        "Maximum training GRID": config["max_train"],
        "GP candidate workers": config["n_jobs"],
        "Matrix threads / worker": protocol["threads"],
        "MLE settings": protocol["mle_options"],
        "Frozen NN SHA256": protocol["bound_model_sha256"],
        "Completed paired replicates": accuracy.replicate.nunique() if not accuracy.empty else 0,
    }
    display(pd.DataFrame({"Setting": conditions.keys(), "Value": [str(v) for v in conditions.values()]}))


## 2. 五係數 RMSE：兩層分開看

每次先對所有 GRID 算 RMSE；多次模擬再報告「各次 RMSE 的平均與標準差」，不把相依格點當成獨立重複實驗。1 次只能試跑，不能評估跨 replicate 的變異。

\[
\mathrm{Improvement}=100\left(1-\frac{\mathrm{mean\ RMSE}_{NN}}{\mathrm{mean\ RMSE}_{MLE}}\right)\%.
\]

完整流程準確度使用 **外層 OOF GP 對已知真值**的 RMSE，不用 GP 對 NN／MLE 的一致性誤差代替。


In [ ]:
COEFFICIENTS = ["mu0", "beta_mu", "eta0", "beta_sigma", "xi0"]
LABELS = [r"$\mu_0$ (°C)", r"$\beta_\mu$ (°C/decade)", r"$\eta_0=\log\sigma_0$",
          r"$\beta_\sigma$ (/decade)", r"$\xi_0$"]
summary = pd.DataFrame()
if not accuracy.empty:
    required = {"replicate", "stage", "coefficient", "RMSE_NN", "RMSE_MLE", "n_grids"}
    if not required.issubset(accuracy.columns):
        raise ValueError("結果欄位不完整")
    summary = accuracy.groupby(["stage", "coefficient"], sort=False).agg(
        n_replicates=("replicate", "count"),
        NN_RMSE=("RMSE_NN", "mean"), MLE_RMSE=("RMSE_MLE", "mean"),
        NN_SD=("RMSE_NN", "std"), MLE_SD=("RMSE_MLE", "std")).reset_index()
    summary["NN_improvement_percent"] = 100 * (
        1 - summary.NN_RMSE / summary.MLE_RMSE.replace(0, np.nan))
    for stage, title in [("estimation", "估計器：NN vs MLE"), ("gp_oof", "完整流程：NN＋GP vs MLE＋GP")]:
        table = summary[summary.stage.eq(stage)].set_index("coefficient").reindex(COEFFICIENTS)
        display(Markdown(f"### {title}"))
        columns = ["n_replicates", "MLE_RMSE", "NN_RMSE", "NN_improvement_percent"]
        if table.n_replicates.max() > 1:
            columns += ["MLE_SD", "NN_SD"]
        display(table[columns].style.format(precision=5, na_rep="—"))


In [ ]:
# 5 個係數 × 2 個評估階段；兩條柱表示方法，不是 train / validation。
if not summary.empty:
    fig, axes = plt.subplots(5, 2, figsize=(11, 15), constrained_layout=True)
    for i, (coef, label) in enumerate(zip(COEFFICIENTS, LABELS)):
        for j, stage in enumerate(["estimation", "gp_oof"]):
            ax = axes[i, j]
            row = summary[(summary.stage == stage) & (summary.coefficient == coef)].iloc[0]
            sd = [row.MLE_SD, row.NN_SD] if row.n_replicates > 1 else None
            ax.bar(["MLE" if j == 0 else "MLE + GP", "BOUND NN" if j == 0 else "BOUND NN + GP"],
                   [row.MLE_RMSE, row.NN_RMSE], yerr=sd, capsize=4,
                   color=["#777777", "#2878A5"], width=.55)
            ax.set_ylabel(f"{label}\nRMSE")
            ax.set_title("Estimation" if j == 0 else "Outer-fold spatial prediction")
            ax.grid(axis="y", alpha=.2); ax.set_axisbelow(True)
    plt.show()


## 3. 時間比較

- **Estimator**：新一次 NN 載入、21D、decoder、逆轉尺度及診斷；或新一次 MLE 配適及診斷。不只計 NN forward。
- **Spatial CV**：各自執行相同 nested buffered FFS／kernel 選擇與 OOF。
- **End-to-end**：估計器開始到 GP、support、RL／真值／CRPS 評分、輸出及雜湊結束。**不含**原始資料前處理、NN 離線訓練、校準真值、生成兩份資料、執行前輸入檢查及最後 report JSON。CRPS 另有獨立計時，不算進 estimator 時間。舊 v1 結果不含 CRPS。
- 使用 wall time；不把平行 MLE 的 fit seconds 加總當整體耗時。
- 不同電腦／套件／執行緒設定的兩條路線不計加速倍數。同設定仍無法控制所有背景負載。
- GP 若占大部分時間，NN 推論快很多也不代表整套同樣快很多。


In [ ]:
if not timing.empty:
    display(timing[["replicate", "timing_comparable", "same_estimator_worker_budget"]])
    if not timing.same_estimator_worker_budget.all():
        display(Markdown("**MLE 使用多 worker：以下是實際資源設定的時間，不是等核心估計器比較。**"))
    comparable = timing[timing.timing_comparable.eq(True)]
    rows = []
    for stage, label in [("estimator", "Estimator"), ("spatial_cv", "Spatial CV"), ("workflow", "End-to-end")]:
        nn = comparable[f"{stage}_nn_seconds"].mean()
        mle = comparable[f"{stage}_mle_seconds"].mean()
        rows.append({"Stage": label, "Comparable pairs": len(comparable),
                     "NN mean seconds": nn, "MLE mean seconds": mle,
                     "Speedup MLE/NN": mle / nn if nn > 0 else np.nan})
    display(pd.DataFrame(rows).style.format(precision=3, na_rep="—"))
    display(Markdown("加速倍數 = MLE 平均 wall time / NN 平均 wall time；大於 1 表示 NN 路線較快。"))


In [ ]:
if not timing.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for ax, stage, title in zip(axes, ["estimator", "workflow"], ["Estimator time", "End-to-end time"]):
        for method, color in [("mle", "#777777"), ("nn", "#2878A5")]:
            ax.plot(timing.replicate, timing[f"{stage}_{method}_seconds"]/60,
                    marker="o", color=color, label=method.upper())
        ax.set_xlabel("Replicate"); ax.set_ylabel("Wall time (minutes)")
        ax.set_title(title); ax.legend(); ax.grid(alpha=.2)
    plt.show()


## 4. Support 與數值品質

NN、MLE 的 observed support 與 **GP OOF support** 分開報告；BOUND 的保證不會自動延伸到 GP。任何 MLE 格點失敗，該次 MLE 空間流程會停下並保存報告，不刪失敗格點再宣稱改善。邊界解、重試與 optimizer 警示也要檢查。


In [ ]:
if not quality.empty:
    q = quality.copy()
    q["estimator_support_failure_percent"] = 100*q.estimator_support_failures/q.n_grids
    q["gp_oof_support_failure_percent"] = 100*q.gp_oof_support_failures/q.n_grids
    display(q.drop(columns=["n_grids"]).style.format(precision=3, na_rep="—"))
# 包括未完成 pair 的 MLE 診斷，避免成功表格掩蓋失敗。
fit_reports = []
for path in sorted((RESULT_ROOT / "replicates").glob("replicate_*/mle/attempt_*/mle_fit_report.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    fit_reports.append({"attempt": str(path.relative_to(RESULT_ROOT)),
                        "requested": r["n_requested"], "failed": r["n_failed"]})
if fit_reports:
    display(Markdown("### MLE attempts（含失敗及重跑，不是獨立 replicate 數）"))
    display(pd.DataFrame(fit_reports))
elif protocol:
    display(Markdown("尚無 MLE fit report；不能解讀為 MLE 失敗數為零。"))


## 5. 指定年份的空間參數圖：五張，每張 3×5

**每張只畫一個參數**；三列為模擬真值、MLE、BOUND NN，五欄是第 10／20／30／40／50 年，對應 1985／1995／2005／2015／2025 年。每個色塊表示一個原始 GRID 中心，不做空間插值，不刪除難估的格點。

五張圖分別為：

1. $\mu(t)=\mu_0+\beta_\mu t_c$；
2. $\beta_\mu$（時間固定）；
3. $\eta(t)=\log\sigma(t)=\eta_0+\beta_\sigma t_c$，**不是** $\sigma(t)$；
4. $\beta_\sigma$（時間固定）；
5. $\xi_0$（時間固定）。

所有估計都使用完整 50 年。這裡不是用前 10／20／30／40／50 年重新估計；$\mu_0$、$\eta_0$ 本身也不是每年的數值。三個固定係數的五欄必然重複，不代表重跑五次模型。

同一參數圖的 15 格共用色階及地理範圍，MLE／NN 各格標註對模擬真值的 RMSE。預設看第 1 個已完成成對 replicate；可切換其他 replicate，但不以「最漂亮的一次」代替所有模擬結果。預設看直接估計；YEAR_PLOT_STAGE 改成 gp_oof 才看完整空間流程的 OOF 結果，標題會同步改變。

輸出為 5 張 PNG（300 dpi）及 5 張 SVG，另存年份 RMSE 與來源雜湊報告，不改原有比較表或模型。


In [ ]:
# 年份圖設定：整份序列仍使用 1976–2025，不做截短重估。
YEAR_PLOT_REPLICATE = 1
YEAR_PLOT_STAGE = "estimation"  # 或 "gp_oof"
SAVE_YEAR_FIGURES = True
if accuracy.empty:
    display(Markdown("尚無成對模擬結果；目前不產生正式年度參數圖，也不拿示範數據代替。"))
else:
    import sys
    for parent in [Path.cwd(), *Path.cwd().parents]:
        source_root = parent / "src"
        if (source_root / "taiwan_simulation/plot_nn_mle_year_slices.py").is_file():
            if str(source_root) not in sys.path:
                sys.path.insert(0, str(source_root))
            break
    from taiwan_simulation.plot_nn_mle_year_slices import load_pair, make_figures, save_figures
    year_pair = load_pair(RESULT_ROOT, replicate=YEAR_PLOT_REPLICATE, stage=YEAR_PLOT_STAGE)
    year_figures, year_metrics, year_metadata = make_figures(year_pair)
    figure_folder = RESULT_ROOT / "figures" / f"replicate_{YEAR_PLOT_REPLICATE:03d}" / YEAR_PLOT_STAGE
    if SAVE_YEAR_FIGURES:
        save_figures(year_figures, year_metrics, year_metadata, figure_folder)
        display(Markdown(f"五張年度參數圖已輸出至：{figure_folder}"))
    for figure in year_figures.values():
        display(figure)
        plt.close(figure)
    display(year_metrics.pivot_table(index=["quantity", "year_number", "year"],
                                     columns="method", values="RMSE"))


## 6. 年最大溫度分布 CRPS

CRPS 比較每個 GRID、每年的 **GEV 累積分布與一筆獨立年最大溫度**，單位 °C，越低越好。不是五係數本身的 RMSE，也不是 GP 係數的常態分布分數。

每次使用同一已知真值生成 A／B 兩份獨立 50 年資料。A 用於 NN、MLE 與 GP，B 只計分；兩方法共用 B。B 的年份仍為 1976–2025，**不是預測未來 50 年**。評分不重新訓練模型，也不改 inner-CV 的選模指標。

$$\mathrm{CRPS}(F,y)=\int_{-\infty}^{\infty}[F(x)-\mathbf{1}\{y\le x\}]^2\,dx.$$

- 分別報告直接估計器與 GP 外層 OOF 的 CRPS，平均涵蓋全部 GRID × 50 年。
- Oracle 用真正生成 GEV 評分同一份 B，作為參考；即使分布正確，CRPS 通常仍大於 0。
- B 在預測 support 外仍照算，不能刪除、截斷觀測或拿 B 修正 decoder／GP。
- 實作限定有限一階矩的 $\xi<1$。不合法參數、未支援 shape 或數值失敗會留下 NaN 與計數；任何一格失敗，該總平均不提供，不用 nanmean 偷刪。這不表示 $\xi\ge1$ 的所有 CDF 積分必然發散。
- 這是 plug-in GEV，未整合係數或 GP 的估計不確定性。CRPS 不取代 support、RL 尾端誤差或空間聯合相依診斷。

公式來源：[Friederichs & Thorarinsdottir (2012), equations 9–10](https://doi.org/10.1002/env.2176)。


In [ ]:
crps = read_csv("crps_comparison.csv")
crps_summary = read_csv("crps_summary.csv")
crps_yearly = read_csv("crps_yearly_comparison.csv")
if crps.empty:
    display(Markdown("**尚無獨立資料的 CRPS 結果。** 舊 v1 沒有此評分，不能用配適資料代替或填入 0；新版 compare-mle 輸出到 v2_crps。"))
else:
    display(crps_summary.style.format(precision=5, na_rep="無法計分"))
    display(crps[["replicate", "stage", "n_total", "n_invalid_NN", "n_invalid_MLE",
                  "n_outside_support_NN", "n_outside_support_MLE"]])
    if crps[["n_invalid_NN", "n_invalid_MLE"]].to_numpy().sum():
        display(Markdown("**有未計分項目：總平均及改善率不能作完整比較。** 請檢查各路線 crps_metrics.csv 的無效參數／shape 計數。"))
    if not timing.empty and {"crps_nn_seconds", "crps_mle_seconds"}.issubset(timing.columns):
        display(timing[["replicate", "crps_nn_seconds", "crps_mle_seconds"]])
    if not crps_yearly.empty:
        def complete_mean(series):
            return series.mean() if series.notna().all() else np.nan
        annual = crps_yearly.groupby(["stage", "year"], sort=True)[
            ["mean_CRPS_C_NN", "mean_CRPS_C_MLE", "oracle_CRPS_C"]].agg(complete_mean).reset_index()
        fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
        for ax, stage in zip(axes, ["estimation", "gp_oof"]):
            part = annual[annual.stage.eq(stage)]
            for column, label in [("mean_CRPS_C_NN", "BOUND NN"),
                                  ("mean_CRPS_C_MLE", "MLE"), ("oracle_CRPS_C", "True GEV reference")]:
                ax.plot(part.year, part[column], label=label)
            ax.set_title("Estimation" if stage == "estimation" else "GP outer-fold predictions")
            ax.set_xlabel("Year"); ax.set_ylabel("Mean CRPS (°C)")
            ax.legend(); ax.grid(alpha=.2)
        plt.show()
        display(annual[annual.year.isin([1985, 1995, 2005, 2015, 2025])])


## 7. 如何下結論

1. 分別描述五係數的估計器／GP OOF RMSE，必要時補 MAE、Bias（已存在 parameter_comparison.csv）。
2. 分別描述估計器／完整流程加速倍數，附 CPU、worker 設定，不把 NN 推論速度外推成整套速度。
3. 同時報告 MLE 失敗、邊界解、support 超界及 GP optimizer 警示。
4. 多次模擬才報告跨 replicate 的變異；目前不自動做顯著性檢定。
5. 真值由 NN＋GP 校準，可能更接近其假設。結論**條件於此生成機制**，不是證明 NN 普遍優於 MLE。後續可加獨立設計曲面、模型失配及時空相關情境。

### 在更新後的「論文跑」執行

需要已完成的 EAS 真實流程作校準來源；通過雜湊檢查的既有校準可沿用。兩路線會新跑相同模擬資料上的估計＋完整 GP，不混入舊計時。

~~~powershell
cd "F:\論文跑"
# 先做 1 次成對比較（兩條完整 GP 流程，可能很久）
powershell -NoProfile -ExecutionPolicy Bypass -File .\run.ps1 -Mode compare-mle -Replicates 1 -Jobs -2 -Threads 1 -MleJobs 1

# 確認後再擴充；總計 100 次，不是再加 100 次
powershell -NoProfile -ExecutionPolicy Bypass -File .\run.ps1 -Mode compare-mle -Replicates 100 -Jobs -2 -Threads 1 -MleJobs 1
~~~

結果：outputs\simulations\eas_calibrated_v1\mle_comparison_v2_crps。舊 v1 結果保留，不混合舊計時。
已完成路線驗證後跳過並保留原計時；未完成者另開 attempt 重跑，不支援折內續算。
D／F 路徑由搬機包位置推導。這些指令需新版程式，舊搬機包不會自動更新。
